In [1]:
import os, sys
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display, Markdown, Math
import numpy as np
import pandas as pd
import networkx as nx
from scipy.stats import norm
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Versión local de la librería: se busca src/pmule.py subiendo desde el directorio actual
_aqui = os.path.abspath('.')
while _aqui != os.path.dirname(_aqui) and not os.path.isfile(os.path.join(_aqui, 'src', 'pmule.py')):
    _aqui = os.path.dirname(_aqui)
sys.path.append(os.path.join(_aqui, 'src'))
from pmule import (PredecessorTable, ProjectGraph, EarnedValue,
                   pinta_tikz, plot_cargas, plot_curvas_valor_ganado,
                   tanto_por_uno, ordena_rutas)

GANTT_PARAMS = dict(arrows=True)   # flechas de prelación en los diagramas de Gantt (rojas las críticas)

In [2]:
# Funciones auxiliares para mostrar los cálculos intermedios paso a paso
from itertools import combinations as _combinations


def detalle_tiempos_nodos(proyecto, duraciones):
    """Tiempos tempranos y tardíos de los nodos con los candidatos que se comparan en cada uno."""
    G = proyecto.pert_graph
    d = {data['activity']: (0 if data['activity'].startswith('@') else int(duraciones[data['activity']]))
         for _, _, data in G.edges(data=True)}
    nodos = list(proyecto.nodes)
    early, filas = {}, []
    for j in nodos:
        entrantes = [(i, data['activity']) for i, _, data in G.in_edges(j, data=True)]
        if not entrantes:
            early[j] = 0
            filas.append({'nodo': j, 'candidatos t_i + d_ij': 'nodo inicial', 'tiempo temprano = máx': 0})
            continue
        cand = {f'{a}: {early[i]} + {d[a]} = {early[i] + d[a]}': early[i] + d[a] for i, a in entrantes}
        early[j] = max(cand.values())
        filas.append({'nodo': j, 'candidatos t_i + d_ij': '; '.join(cand), 'tiempo temprano = máx': early[j]})
    tempranos = pd.DataFrame(filas).set_index('nodo')
    late, filas = {}, []
    for i in reversed(nodos):
        salientes = [(j, data['activity']) for _, j, data in G.out_edges(i, data=True)]
        if not salientes:
            late[i] = early[i]
            filas.append({'nodo': i, 'candidatos t*_j − d_ij': 'nodo final (= su tiempo temprano)', 'tiempo tardío = mín': late[i]})
            continue
        cand = {f'{a}: {late[j]} − {d[a]} = {late[j] - d[a]}': late[j] - d[a] for j, a in salientes}
        late[i] = min(cand.values())
        filas.append({'nodo': i, 'candidatos t*_j − d_ij': '; '.join(cand), 'tiempo tardío = mín': late[i]})
    tardios = pd.DataFrame(filas).set_index('nodo')
    return tempranos, tardios


def detalle_holguras(calendario, prelaciones):
    """Cómo se obtiene cada fecha y cada holgura a partir de las vecinas."""
    pred = prelaciones.immediate_predecessor
    suc = {a: sorted(s for s, ps in pred.items() if a in ps) for a in calendario.index}
    fin = int(calendario['fin_mas_temprano'].max())
    filas = []
    for a in calendario.index:
        ES, EF = int(calendario.loc[a, 'inicio_mas_temprano']), int(calendario.loc[a, 'fin_mas_temprano'])
        LS, LF = int(calendario.loc[a, 'inicio_mas_tardio']), int(calendario.loc[a, 'fin_mas_tardio'])
        dur = int(calendario.loc[a, 'duracion'])
        ps, ss = sorted(pred[a]), suc[a]
        ef_pred = {p: int(calendario.loc[p, 'fin_mas_temprano']) for p in ps}
        es_suc = {s: int(calendario.loc[s, 'inicio_mas_temprano']) for s in ss}
        lf_pred = {p: int(calendario.loc[p, 'fin_mas_tardio']) for p in ps}
        ls_suc = {s: int(calendario.loc[s, 'inicio_mas_tardio']) for s in ss}
        min_es_suc = min(es_suc.values()) if es_suc else fin
        max_lf_pred = max(lf_pred.values()) if lf_pred else 0
        filas.append({
            'actividad': a,
            'ES = máx EF pred.': (f"máx({', '.join(f'{p}: {v}' for p, v in ef_pred.items())}) = {ES}" if ps else f'inicial → {ES}'),
            'EF = ES + d': f'{ES} + {dur} = {EF}',
            'LF = mín LS suc.': (f"mín({', '.join(f'{s}: {v}' for s, v in ls_suc.items())}) = {LF}" if ss else f'final → {LF}'),
            'LS = LF − d': f'{LF} − {dur} = {LS}',
            'H_total = LS − ES': f'{LS} − {ES} = {LS - ES}',
            'H_libre = mín ES suc. − EF': f"{min_es_suc}{'' if ss else ' (fin)'} − {EF} = {min_es_suc - EF}",
            'H_indep. = máx(0, mín ES suc. − máx LF pred. − d)': f'máx(0, {min_es_suc} − {max_lf_pred} − {dur}) = {max(0, min_es_suc - max_lf_pred - dur)}',
        })
    return pd.DataFrame(filas).set_index('actividad')


def detalle_rutas(R, valores, nombre='duración', decimales=0):
    """Suma término a término de un valor (duraciones, varianzas…) a lo largo de cada ruta."""
    filas = {}
    for r in R.index:
        acts = list(R.columns[R.loc[r] == 1])
        total = float(sum(valores[a] for a in acts))
        filas[r] = {'actividades': ' - '.join(acts),
                    f'{nombre} = suma': ' + '.join(f'{a}({valores[a]:.{decimales}f})' for a in acts) + f' = {total:.{decimales}f}',
                    nombre: total}
    return pd.DataFrame(filas).T


def detalle_estimaciones(datos, a='a', m='m', b='b'):
    """Media y varianza PERT de cada actividad con los números sustituidos."""
    filas = {}
    for act, fila in datos.iterrows():
        A, M, B = float(fila[a]), float(fila[m]), float(fila[b])
        filas[act] = {'D = (a + 4m + b)/6': f'({A:g} + 4·{M:g} + {B:g})/6 = {(A + 4 * M + B) / 6:.2f}',
                      'σ² = ((b − a)/6)²': f'(({B:g} − {A:g})/6)² = {((B - A) / 6) ** 2:.3f}'}
    return pd.DataFrame(filas).T


def detalle_ackoff(proyecto, duraciones, topes, cur, pasos):
    """En cada paso de Ackoff-Sasieni: rutas críticas, opciones consideradas (actividades comunes y parejas)
    con su coste, y la elegida."""
    R = proyecto.path_matrix(dummies=False)
    d = duraciones.astype(float).copy()
    filas = []
    for paso, elegidas in sorted(pasos.items()):
        L = R @ d[R.columns]
        T = float(L.max())
        criticas = [r for r in L.index if np.isclose(L[r], T)]
        conjuntos = [set(R.columns[R.loc[r] == 1]) for r in criticas]
        con_margen = {x for x in R.columns if d[x] > topes[x] and np.isfinite(cur[x])}
        opciones = {x: float(cur[x]) for x in set.intersection(*conjuntos) & con_margen}
        if len(conjuntos) > 1:
            cand = sorted(set().union(*conjuntos) & con_margen)
            for p, q in _combinations(cand, 2):
                if all(p in s or q in s for s in conjuntos) and p not in opciones and q not in opciones:
                    opciones[f'{p}+{q}'] = float(cur[p] + cur[q])
        orden = sorted(opciones.items(), key=lambda kv: kv[1])
        filas.append({'paso': paso + 1, 'duración': int(T), 'rutas críticas': ', '.join(criticas),
                      'opciones (coste)': '; '.join(f'{k} ({v:.0f})' for k, v in orden[:6]) + (' …' if len(orden) > 6 else ''),
                      'elegida': ', '.join(elegidas), 'coste del paso': float(sum(cur[x] for x in elegidas))})
        for x in elegidas:
            d[x] -= 1
    return pd.DataFrame(filas).set_index('paso')


def detalle_valor_ganado(datos_vg, calendario, semana):
    """PV, EV y AC actividad a actividad en el punto de control."""
    filas = {}
    for a, fila in datos_vg.iterrows():
        es, dur = int(calendario.loc[a, 'inicio_mas_temprano']), int(calendario.loc[a, 'duracion'])
        frac = min(max(semana - es, 0), dur) / dur
        filas[a] = {'BAC': float(fila['BAC']), 'planificado hasta la semana': f'{min(max(semana - es, 0), dur)} de {dur}',
                    'PV = BAC·fracción': f'{fila["BAC"]:.0f}·{frac:.2f} = {fila["BAC"] * frac:.1f}',
                    'realizado (%)': int(fila['performed']),
                    'EV = BAC·%': f'{fila["BAC"]:.0f}·{fila["performed"] / 100:.2f} = {fila["BAC"] * fila["performed"] / 100:.1f}',
                    'AC': float(fila['AC']), 'PV_i': fila['BAC'] * frac, 'EV_i': fila['BAC'] * fila['performed'] / 100}
    t = pd.DataFrame(filas).T
    return t


def muestra_formula(nombre, expresion, valor, unidades=''):
    """Una línea «nombre = expresión = valor» en Markdown."""
    return Markdown(f'**{nombre}** = {expresion} = **{valor}**{(" " + unidades) if unidades else ""}')

In [3]:
%%capture
%mkdir -p ~/temp
%cd ~/temp

# Enunciado

Para cada afirmación diga si es verdadera o falsa. Si es falsa, dé un contraejemplo concreto (puede apoyarse en el proyecto cuyo cuadro se adjunta, o en otro); si es verdadera, justifíquela brevemente.

In [4]:
pregunta_1 = "1. «Si una actividad no crítica se retrasa más que su holgura total, el proyecto se retrasa.»  "
pregunta_2 = "2. «La suma de las holguras libres de las actividades de una ruta es como mucho el margen de esa ruta (τ menos su duración).»  "
pregunta_3 = "3. «La holgura independiente de una actividad puede ser mayor que su holgura libre.»  "
pregunta_4 = "4. «Una actividad con holgura libre cero es crítica.»  "
pregunta_5 = "5. «La holgura total de una actividad nunca es menor que su holgura libre.»  "
pregunta_6 = "6. «Si una actividad crítica se retrasa una semana, el proyecto se retrasa una semana.»  "

for item in sorted(dir()):
    if item.startswith('pregunta_'):
        display(Markdown(eval(item)))

1. «Si una actividad no crítica se retrasa más que su holgura total, el proyecto se retrasa.»  

2. «La suma de las holguras libres de las actividades de una ruta es como mucho el margen de esa ruta (τ menos su duración).»  

3. «La holgura independiente de una actividad puede ser mayor que su holgura libre.»  

4. «Una actividad con holgura libre cero es crítica.»  

5. «La holgura total de una actividad nunca es menor que su holgura libre.»  

6. «Si una actividad crítica se retrasa una semana, el proyecto se retrasa una semana.»  

In [5]:
#| tbl-cap: "Proyecto de apoyo"
#| label: tbl-datosEnunciado
datos_enunciado = pd.DataFrame([
#  activity, predecessor, duration
    ('A'     , '---'      , 5       ),
    ('B'     , '---'      , 1       ),
    ('C'     , 'A,B'      , 4       ),
    ('D'     , 'C'        , 1       ),
    ('E'     , 'B'        , 1       ),
    ('F'     , 'D,E'      , 6       ),
    ('G'     , 'E'        , 6       ),
    ('H'     , 'D,G'      , 6       ),
    ('I'     , 'F,H'      , 6       ),
], columns=['activity', 'predecessor', 'duration']).set_index('activity')
datos_enunciado

,predecessor,duration
activity,,
A,---,5
B,---,1
C,"A,B",4
D,C,1
E,B,1
F,"D,E",6
G,E,6
H,"D,G",6
I,"F,H",6


El proyecto corresponde a la instalación de una planta fotovoltaica sobre cubierta; las actividades son las siguientes (las duraciones del cuadro están en días).

In [6]:
#| tbl-cap: "Descripción de las actividades"
#| label: tbl-descripciones
descripciones = pd.Series({
    'A': 'Estudio de sombras y producción',
    'B': 'Cálculo estructural de la cubierta',
    'C': 'Tramitación de permisos',
    'D': 'Acopio de módulos e inversores',
    'E': 'Montaje de la estructura soporte',
    'F': 'Colocación de módulos',
    'G': 'Cableado de corriente continua',
    'H': 'Instalación de inversores',
    'I': 'Cuadro de protecciones y contador',
}, name='Descripción')
descripciones.to_frame()

,Descripción
A,Estudio de sombras y producción
B,Cálculo estructural de la cubierta
C,Tramitación de permisos
D,Acopio de módulos e inversores
E,Montaje de la estructura soporte
F,Colocación de módulos
G,Cableado de corriente continua
H,Instalación de inversores
I,Cuadro de protecciones y contador


# Resolución

In [7]:
datos_proyecto = datos_enunciado.copy()
prelaciones = PredecessorTable.from_dataframe_of_strings(datos_proyecto, activity='activity', predecessor='predecessor')
proyecto = prelaciones.create_project()
calendario = proyecto.calendar(datos_proyecto['duration'])
duracion = proyecto.duration(datos_proyecto['duration'])
rutas_criticas = proyecto.critical_path(datos_proyecto['duration'])
R = proyecto.path_matrix(dummies=False)
calendario

,inicio_mas_temprano,inicio_mas_tardio,fin_mas_temprano,fin_mas_tardio,H_libre,H_independiente,H_total,duracion
activity,,,,,,,,
A,0,0,5,5,0,0,0,5
B,0,2,1,3,0,0,2,1
C,5,5,9,9,0,0,0,4
D,9,9,10,10,0,0,0,1
E,1,3,2,4,0,0,2,1
F,10,10,16,16,0,0,0,6
G,2,4,8,10,2,0,2,6
H,10,10,16,16,0,0,0,6
I,16,16,22,22,0,0,0,6


## Apartado 1

In [8]:
display(Markdown(pregunta_1))

1. «Si una actividad no crítica se retrasa más que su holgura total, el proyecto se retrasa.»  

«Si una actividad no crítica se retrasa más que su holgura total, el proyecto se retrasa.» — **Verdadera**.

In [9]:
a = calendario.index[calendario['H_total'] > 0][0]
h = int(calendario.loc[a, 'H_total'])
d2 = datos_proyecto['duration'].copy(); d2[a] += h + 1
Markdown(f'Justificación: la holgura total es exactamente el retraso que admite sin mover el fin del proyecto. Comprobación con {a} (holgura {h}): retrasada {h + 1} semanas el proyecto pasa de {duracion} a {proyecto.duration(d2)}.')

Justificación: la holgura total es exactamente el retraso que admite sin mover el fin del proyecto. Comprobación con B (holgura 2): retrasada 3 semanas el proyecto pasa de 22 a 23.

## Apartado 2

In [10]:
display(Markdown(pregunta_2))

2. «La suma de las holguras libres de las actividades de una ruta es como mucho el margen de esa ruta (τ menos su duración).»  

«La suma de las holguras libres de las actividades de una ruta es como mucho el margen de esa ruta (τ menos su duración).» — **Verdadera**.

In [11]:
filas = []
for r in R.index:
    acts = list(R.columns[R.loc[r] == 1])
    filas.append({'ruta': r, 'suma de holguras libres': int(calendario.loc[acts, 'H_libre'].sum()), 'margen de la ruta': int(duracion - (R.loc[r] * datos_proyecto['duration'][R.columns]).sum())})
display(pd.DataFrame(filas).set_index('ruta'))
Markdown('Justificación: las holguras libres de una ruta son huecos disjuntos entre el fin de una actividad y el inicio de la siguiente (o el fin del proyecto); su suma es como mucho el margen total de la ruta, y es menor si alguna actividad de la ruta empieza después del fin de su predecesora en la ruta por culpa de otra predecesora.')

,suma de holguras libres,margen de la ruta
ruta,,
Route_1,0,0
Route_2,0,0
Route_3,0,4
Route_4,0,4
Route_5,0,8
Route_6,2,2
Route_7,0,8


Justificación: las holguras libres de una ruta son huecos disjuntos entre el fin de una actividad y el inicio de la siguiente (o el fin del proyecto); su suma es como mucho el margen total de la ruta, y es menor si alguna actividad de la ruta empieza después del fin de su predecesora en la ruta por culpa de otra predecesora.

## Apartado 3

In [12]:
display(Markdown(pregunta_3))

3. «La holgura independiente de una actividad puede ser mayor que su holgura libre.»  

«La holgura independiente de una actividad puede ser mayor que su holgura libre.» — **Falsa**.

In [13]:
Markdown(f'Justificación: H_indep ≤ H_libre ≤ H_total siempre. La libre es la parte de la total que no afecta a las sucesoras (H_libre = mín ES suc − EF ≤ LS − ES = H_total, porque mín ES suc ≤ mín LS suc = LF); la independiente, además, supone las predecesoras acabando lo más tarde posible, lo que sólo puede reducir el margen. En el proyecto: ' + ', '.join(f"{a}: {int(r.H_independiente)} ≤ {int(r.H_libre)} ≤ {int(r.H_total)}" for a, r in calendario.iterrows()) + '.')

Justificación: H_indep ≤ H_libre ≤ H_total siempre. La libre es la parte de la total que no afecta a las sucesoras (H_libre = mín ES suc − EF ≤ LS − ES = H_total, porque mín ES suc ≤ mín LS suc = LF); la independiente, además, supone las predecesoras acabando lo más tarde posible, lo que sólo puede reducir el margen. En el proyecto: A: 0 ≤ 0 ≤ 0, B: 0 ≤ 0 ≤ 2, C: 0 ≤ 0 ≤ 0, D: 0 ≤ 0 ≤ 0, E: 0 ≤ 0 ≤ 2, F: 0 ≤ 0 ≤ 0, G: 0 ≤ 2 ≤ 2, H: 0 ≤ 0 ≤ 0, I: 0 ≤ 0 ≤ 0.

## Apartado 4

In [14]:
display(Markdown(pregunta_4))

4. «Una actividad con holgura libre cero es crítica.»  

«Una actividad con holgura libre cero es crítica.» — **Falsa**.

In [15]:
ejemplo = calendario.index[(calendario['H_libre'] == 0) & (calendario['H_total'] > 0)].tolist()
Markdown(f'Contraejemplo: {", ".join(ejemplo)} tiene(n) holgura libre 0 y holgura total {", ".join(str(int(calendario.loc[a, "H_total"])) for a in ejemplo)} > 0: '
         'no es crítica, sólo ocurre que su sucesora empieza justo cuando ella termina (en el calendario de inicios tempranos).')

Contraejemplo: B, E tiene(n) holgura libre 0 y holgura total 2, 2 > 0: no es crítica, sólo ocurre que su sucesora empieza justo cuando ella termina (en el calendario de inicios tempranos).

## Apartado 5

In [16]:
display(Markdown(pregunta_5))

5. «La holgura total de una actividad nunca es menor que su holgura libre.»  

«La holgura total de una actividad nunca es menor que su holgura libre.» — **Verdadera**.

In [17]:
Markdown(f'Justificación: H_indep ≤ H_libre ≤ H_total siempre. La libre es la parte de la total que no afecta a las sucesoras (H_libre = mín ES suc − EF ≤ LS − ES = H_total, porque mín ES suc ≤ mín LS suc = LF); la independiente, además, supone las predecesoras acabando lo más tarde posible, lo que sólo puede reducir el margen. En el proyecto: ' + ', '.join(f"{a}: {int(r.H_independiente)} ≤ {int(r.H_libre)} ≤ {int(r.H_total)}" for a, r in calendario.iterrows()) + '.')

Justificación: H_indep ≤ H_libre ≤ H_total siempre. La libre es la parte de la total que no afecta a las sucesoras (H_libre = mín ES suc − EF ≤ LS − ES = H_total, porque mín ES suc ≤ mín LS suc = LF); la independiente, además, supone las predecesoras acabando lo más tarde posible, lo que sólo puede reducir el margen. En el proyecto: A: 0 ≤ 0 ≤ 0, B: 0 ≤ 0 ≤ 2, C: 0 ≤ 0 ≤ 0, D: 0 ≤ 0 ≤ 0, E: 0 ≤ 0 ≤ 2, F: 0 ≤ 0 ≤ 0, G: 0 ≤ 2 ≤ 2, H: 0 ≤ 0 ≤ 0, I: 0 ≤ 0 ≤ 0.

## Apartado 6

In [18]:
display(Markdown(pregunta_6))

6. «Si una actividad crítica se retrasa una semana, el proyecto se retrasa una semana.»  

«Si una actividad crítica se retrasa una semana, el proyecto se retrasa una semana.» — **Verdadera**.

In [19]:
a = calendario.index[calendario['H_total'] == 0][0]
d2 = datos_proyecto['duration'].copy(); d2[a] += 1
Markdown(f'Justificación: una actividad crítica está en una ruta de duración τ; alargarla x alarga esa ruta a τ + x, y ninguna ruta puede ser más corta que la más larga. Comprobación con {a}: {duracion} → {proyecto.duration(d2)}.')

Justificación: una actividad crítica está en una ruta de duración τ; alargarla x alarga esa ruta a τ + x, y ninguna ruta puede ser más corta que la más larga. Comprobación con A: 22 → 23.